# **CSV Comparison Runner**

## **1. Project Setup**

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## **2. Imports**

In [ ]:
from pathlib import Path

from src.csv_comparison import compare_csv_files, print_comparison_summary
from src.comparison_exports import save_comparison_result, print_exported_comparison_paths

## **3. Comparison Settings**

In [ ]:
# ---------------------------------------------------------------------------
# Comparison Settings
# ---------------------------------------------------------------------------

COMPARISON_NAME = "session_comparison"

# Choose which processed session folders to compare.
SELECTED_SESSIONS = [
    "new_york",
    "london",
    "asia",
    # "all_sessions",
]

# If True, the notebook will automatically use the newest CSV
# from each selected processed session folder.
USE_LATEST_FILE_FROM_EACH_FOLDER = True

# Manual mode.
# Use this if you want to compare exact CSV files.
MANUAL_CSV_PATHS = [
    # PROJECT_ROOT / "data" / "processed" / "new_york" / "your_file.csv",
    # PROJECT_ROOT / "data" / "processed" / "london" / "your_file.csv",
]

MANUAL_LABELS = [
    # "NY file",
    # "London file",
]

SAVE_OUTPUT = True

## **4. Select CSV Files**

In [ ]:
def get_latest_csv(folder: Path) -> Path | None:
    csv_files = sorted(
        folder.glob("*.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    if not csv_files:
        return None

    return csv_files[0]


csv_paths = []
labels = []

if USE_LATEST_FILE_FROM_EACH_FOLDER:
    for session in SELECTED_SESSIONS:
        folder = PROJECT_ROOT / "data" / "processed" / session
        latest_csv = get_latest_csv(folder)

        if latest_csv is None:
            print(f"No CSV found for session: {session} in {folder}")
            continue

        csv_paths.append(latest_csv)
        labels.append(session)

else:
    csv_paths = [Path(path) for path in MANUAL_CSV_PATHS]
    labels = MANUAL_LABELS


print("Selected CSV files:")

for label, path in zip(labels, csv_paths):
    print(f"{label}: {path}")

if len(csv_paths) < 2:
    raise RuntimeError("Select at least two CSV files to compare.")

## **5. Run Comparison**

In [ ]:
comparison_result = compare_csv_files(
    csv_paths=csv_paths,
    labels=labels,
)

print_comparison_summary(comparison_result)

## **6. Regime Summary**

In [ ]:
regime_summary = comparison_result["regime_summary"]
regime_summary

## **7. Pairwise Similarity**

In [ ]:
pairwise_comparisons = comparison_result["pairwise_comparisons"]
pairwise_comparisons

## **8. Save Comparison Results**

In [ ]:
if SAVE_OUTPUT:
    exported_paths = save_comparison_result(
        result=comparison_result,
        comparison_name=COMPARISON_NAME,
        include_timestamp=True,
    )

    print_exported_comparison_paths(exported_paths)
else:
    print("SAVE_OUTPUT is False. Nothing was saved.")